# Laden der Fashion MNIST Daten


In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import fashion_mnist

# Datensatz laden
(x_train, y_train), (x_test, y_test_original) = fashion_mnist.load_data()

# Pixelwerte auf den Bereich [0, 1] skalieren
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Convolution Layer arbeiten besser, auf echten Bildern mit Farbkanal
# --> Wir fügen den Farbkanal hinzu
print("Ohne Kanal:", x_train.shape)
x_train = x_train[..., None]
x_test = x_test[..., None]
print("Mit Kanal:", x_train.shape)

# One Hot encoding
y_train = to_categorical(y_train)
y_test = to_categorical(y_test_original)

Dieses Mal lassen wir die Bilder im 28x28 Format.

# CNN erzeugen

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense

model = Sequential()

model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(64, kernel_size=(3, 3), activation="relu"))

model.add(Flatten())
model.add(Dense(128, activation="relu"))
model.add(Dense(10, activation="softmax"))

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
# Funktioniert wegen eines Versionskonflikts aktuell leider nicht

#import visualkeras
#visualkeras.layered_view(model, legend=True)

# Neuronales Netz trainieren

In [ ]:
model.fit(
    x_train, y_train,
    epochs=12,
    batch_size=128,
    validation_split=0.1
)


# Genauigkeit auf Testdaten

In [ ]:
  model.evaluate(x_test, y_test)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model

# ------------------------------------------------------------
# 1. Filtermasken (Kernel) der Convolution-Schicht auslesen
# ------------------------------------------------------------
conv_layer = model.layers[0]              # erste Conv2D-Schicht
weights, biases = conv_layer.get_weights()

# weights hat die Form (kernel_h, kernel_w, in_channels, n_filter)
print("Shape der Filter:", weights.shape)   # (3, 3, 1, 64)
print("Shape der Biases:", biases.shape)    # (64,)

# Einzelne Filtermaske als 3x3-Matrix
print("Filter 0:\n", weights[:, :, 0, 0])

In [ ]:
# ------------------------------------------------------------
# 2. Filtermasken visualisieren
# ------------------------------------------------------------
n_filter = weights.shape[-1]
n_show = 16                                # erste 16 Filter zeigen

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    kernel = weights[:, :, 0, i]
    ax.imshow(kernel, cmap="gray")
    ax.set_title(f"Filter {i}", fontsize=8)
    ax.axis("off")
plt.suptitle("Gelernte 3x3 Filtermasken")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 3. Merkmalskarten: Filter auf ein MNIST-Bild anwenden#
# ------------------------------------------------------------
feature_model = Model(inputs=model.inputs, outputs=conv_layer.output)

img = x_test[0]                            # Form (28, 28, 1)
batch = img[None, ...]                     # Form (1, 28, 28, 1)

feature_maps = feature_model.predict(batch, verbose=0)
print("Shape der Merkmalskarten:", feature_maps.shape)   # (1, 26, 26, 64)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.set_title(f"Map {i}", fontsize=8)
    ax.axis("off")
plt.suptitle("Merkmalskarten nach Conv2D (inkl. ReLU)")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 4. Ein Filter auf mehrere MNIST-Bilder anwenden
# ------------------------------------------------------------
f = 0
n_bilder = 5
maps = feature_model.predict(x_test[:n_bilder], verbose=0)

fig, axes = plt.subplots(2, n_bilder, figsize=(2.2*n_bilder, 5))
for i in range(n_bilder):
    axes[0, i].imshow(x_test[i][:, :, 0], cmap="gray")
    axes[0, i].set_title(f"Label {y_test[i].argmax()}", fontsize=9)
    axes[1, i].imshow(maps[i, :, :, f], cmap="viridis")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
plt.suptitle(f"Wirkung von Filter {f} auf verschiedene Bilder")
plt.tight_layout()
plt.savefig("mnist_aktivierung.svg", format="svg", bbox_inches="tight")
plt.show()